# DUK leg kinematics

This notebook uses one canonical robot frame. **Only A is fixed at (0, 0).** Every other point is produced by geometry equations.

- +Ox points right, +Oy points up, counter-clockwise is positive.
- D→A = -30°, therefore A→D is the opposite direction.
- `q` is the directed counter-clockwise angle from A→B to D→A.
- `θ_AB = θ_DA - q`.
- D is derived from AD and `θ_AD`; B is derived from AB and `θ_AB`.
- C is the physical circle-intersection solution selected by named topology.
- H/O are derived from the moving rigid BCO basis.
- No mirror, solver-frame rotation or display correction is applied after solving.


In [ ]:
from pathlib import Path
import math

from duk_core.modeling import load_leg_geometry_json, solve_leg_pose, leg_jacobian, sample_leg_trajectory
from duk_core.visualization import draw_leg_linkage, draw_point_trajectories, plot_point_coordinates, plot_point_jacobian

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'robot/config/physical/duk_leg_geometry.json').exists():
            return candidate
    raise FileNotFoundError('Run the notebook from a DUK repository checkout')

repo_root = find_repo_root()
geometry = load_leg_geometry_json(repo_root / 'robot/config/physical/duk_leg_geometry.json')
print(f'DA = {math.degrees(geometry.da_angle_rad):+.1f}°')
print(f'AD = {math.degrees(geometry.ad_angle_rad):+.1f}°')
print('D is derived from A, AD and θ_AD =', geometry.d_point(), 'mm')
geometry


In [ ]:
lower, upper = geometry.geometric_linkage_angle_limits_rad()
print(f'BO = {geometry.bo_mm:.3f} mm')
print(f'CO = {geometry.co_mm:.3f} mm')
print(f'geometric q range = {math.degrees(lower):.3f} .. {math.degrees(upper):.3f}°')
print('q is the directed CCW angle from A→B to D→A.')
print('The endpoints are geometric singular boundaries, not operating limits.')


In [ ]:
q_deg = 75.0
q = math.radians(q_deg)
pose = solve_leg_pose(geometry, q)
jac = leg_jacobian(geometry, pose)
theta_ab_deg = math.degrees(geometry.ab_angle_rad(q))
print(f'q = {q_deg:.2f}°')
print(f'derived A→B direction = {theta_ab_deg:+.2f}° from +Ox')
for name in ('A', 'B', 'C', 'D', 'H', 'O'):
    print(name, '=', pose.point(name), 'mm')
print('dO/dq =', jac.d_o, 'mm/rad')
draw_leg_linkage(pose, geometry).set_title(f'DUK leg — q={q_deg:.1f}°')


In [ ]:
margin = math.radians(2.0)
poses = sample_leg_trajectory(geometry, lower + margin, upper - margin, 180)
draw_point_trajectories(poses).set_title('B/C/H/O trajectories — canonical robot frame')


In [ ]:
plot_point_coordinates(poses, 'O').set_title('Wheel-center coordinates vs q')


In [ ]:
plot_point_jacobian(geometry, poses, 'O').set_title('Wheel-center Jacobian dO/dq')
